# 09 - Repeated small replay corruption on a stability-screened schedule (RCS)
Pre-registration: `experiments/prereg_rcs.md`, committed before any RCS code or run. Code: `s13_rcs.py`, which reuses
`s12_repeated.py`'s run, summarize and test code with its own layout.

**This is a new experiment, separate from RC (notebook 08).** RC stopped at its clean pilot (`results/08_rc/`):
- its 6-update schedule was not stable without poison;
- the SNMP and MSSQL updates made the model forget LDAP.

RCS first has to **show a stable clean trajectory**, and only then are poisoned runs allowed.

| Stage | What it runs | Rule |
|---|---|---|
| S, screen (cell 3) | Clean arm only, seeds 42-44, validation only. Two candidates fixed in priority order: `rcs6` = MSSQL, NTP, SNMP, SSDP, TFTP, LDAP (LDAP last); `rcs5` = the same without LDAP | Select the first candidate that is **stable in 3/3 seeds** |
| C, confirm (cell 4) | The selected schedule, clean arm only, **fresh seeds 45-47** | **PROCEED only if it is stable in 3/3 seeds** |
| P, poisoned (cell 5) | Only if PROCEED. Before every update the attacker flips rho x 2,000 more stored Syn rows to BENIGN (rho 0.10, then 0.05); flips persist. Arms: clean (the confirmation runs, not retrained), poisoned, poisoned_repair (soft R1), and a forced soft-R1 diagnostic | Fixed horizon; rho 0.05 is skipped if the T4 time is over 90 min |

**Stable run** means all three hold:
- every update is accepted by the gate;
- every update's smallest gate margin is >= 0.01;
- Syn val recall stays within 0.02 of A0's.

**If cell 3 or 4 says STOP, do not run cell 5.** Download `rcs_clean.zip` (from cell 4, or written by cell 3 on a
stop) and send it. Nothing is changed after a stop.

**Inputs (Add Input).** Notebook 04's output, as for notebooks 05-08: `nids_out/` with `clean_*.parquet` and
`runs/cii/all12_cii_s42`. Settings: **Internet ON**, accelerator **GPU T4**.

**Expected time:** about 25-35 min of T4. Run cells one at a time, never with Save & Run All.

**If the session dies:** rerun cells 1 to the last unfinished cell. Every stage resumes: finished runs are skipped,
and a stage that already decided prints its saved decision.

In [ ]:
# 1. Locate code and the notebook-04 outputs (read only); everything is written under /kaggle/working/nids_out
import glob, json, os, subprocess, sys, time, zipfile
from pathlib import Path
import pandas as pd
pd.set_option("display.width", 220); pd.set_option("display.max_columns", 40)

CLONE = Path("/kaggle/working/code")
if not CLONE.exists() and os.environ.get("NIDS_CLONE", "1") == "1":
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/DevChi1l/cicddos2019-continual-nids",
                    str(CLONE)], check=False)
CODE_DIR = os.environ.get("NIDS_CODE") or next(
    (str(Path(p).parent) for p in [str(CLONE / "s13_rcs.py")] +
     sorted(glob.glob("/kaggle/input/**/s13_rcs.py", recursive=True)) if Path(p).exists()), None)
assert CODE_DIR, "code with s13_rcs.py not found: turn Internet ON"
sys.path.insert(0, CODE_DIR)
import s5_explain as X
git = subprocess.run(["git", "-C", CODE_DIR, "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip()
print("code:", CODE_DIR, "| git commit:", git)

SRC = X.find_dir("runs/cii/all12_cii_s42/config.json", "NIDS_RC_SRC")
assert SRC and (SRC / "clean_train.parquet").exists(), \
    "attach notebook 04's output: nids_out/ with clean_*.parquet and runs/cii/all12_cii_s42"
WORK = Path(os.environ.get("NIDS_WORK", "/kaggle/working/nids_out"))
RCS = WORK / "rcs"
print("source (read only):", SRC, "\nwrites:", WORK)

def run(*args):
    cmd = [sys.executable, str(Path(CODE_DIR) / "s13_rcs.py"), *map(str, args)]
    print("$", " ".join(cmd[1:]), flush=True)
    subprocess.run(cmd, check=True, cwd=CODE_DIR)

def clean_zip():
    # validation files of stages S and C (no models, no test): send this after a STOP
    run("zip-val", *COMMON, "--bundle", WORK.parent / "rcs_clean.zip")

def show_stage(per_seed):
    rows = [{"seed": s, "t": u["t"], "new": u["new_class"], "accepted": u["accepted"],
             "min_margin": round(u["min_margin"], 3), "binding": u["binding"], "Syn": round(u["Syn_recall"], 4)}
            for s, v in per_seed.items() for u in v["updates"]]
    display(pd.DataFrame(rows).pivot_table(index="seed", columns=["t", "new"], values=["accepted", "min_margin"],
                                           aggfunc="first"))
    for s, v in per_seed.items():
        print(f"  seed {s}: stable={v['stable']} | all accepted={v['all_accepted']} | "
              f"min margin={v['min_margin']:.3f} | max |Syn - A0|={v['max_abs_Syn_change']:.4f}")

COMMON = ["--work", WORK]

In [ ]:
# 2. GPU check; copy the Stage 1 files (hash-checked against all12) and build Stage 1b of both candidate tags
#    (about 1 min). Refuses if the protected val rows differ from the registered feasibility check.
#    In a RESTORE session run this AFTER cell 8.
import torch
assert torch.cuda.is_available(), "Settings -> Accelerator -> GPU T4"
print(torch.cuda.get_device_name(0), torch.__version__)
run("prepare", "--src", SRC, *COMMON)

In [ ]:
# 3. Stage S (screen): clean arm only, both candidates, seeds 42-44, validation only (about 6 min).
t0 = time.time()
run("screen", *COMMON)
s = json.loads((RCS / "screen.json").read_text())
for tag, r in s["candidates"].items():
    print(f"\n{tag} = {r['classes']}: stable in {r['stable_seeds']}/3 seeds")
    show_stage(r["per_seed"])
print(f"\nSELECTED = {s['selected']} | {time.time() - t0:.0f} s")
if not s["selected"]:
    clean_zip()
    print("STOP HERE: no candidate is stable. Send rcs_clean.zip; do not run cells 4-7 (do not change the design)")

In [ ]:
# 4. Stage C (confirmation): the selected schedule, clean arm only, fresh seeds 45-47 (about 3 min).
#    Writes rcs/go.json; the poisoned runs (cell 5) refuse unless it says proceed.
t0 = time.time()
run("confirm", *COMMON)
g = json.loads((RCS / "go.json").read_text())
show_stage(g["per_seed"])
print(f"\nPROCEED = {g['proceed']} ({g['reason']}, tag {g['tag']}) | {time.time() - t0:.0f} s")
clean_zip()
if not g["proceed"]:
    print("STOP HERE: the clean schedule is not stable on fresh seeds. Send rcs_clean.zip; do not run cells 5-7")

In [ ]:
# 5. Stage P (poisoned runs), validation only: rho 0.10 (resumes the confirmation runs), then rho 0.05 unless the
#    measured T4 time > 90 min (about 15-20 min). Re-running this cell resumes; finished runs are skipped.
t0 = time.time()
run("run", *COMMON)
print(f"run: {time.time() - t0:.0f} s")

In [ ]:
# 6. Trajectories and the registered predictions, from saved files (validation only)
run("summarize", *COMMON)
traj = pd.read_csv(RCS / "trajectory_val.csv")
print("deployed model's Syn val recall after update t:")
display(traj.pivot_table(index=["rho", "seed", "arm"], columns="t", values="Syn_recall").round(3))
print("C1 decision per update (True = accepted):")
display(traj[traj.t > 0].pivot_table(index=["rho", "seed", "arm"], columns="t", values="c1_accepted", aggfunc="first"))

## STOP - download BOTH zips from the Output panel (cell 7 writes them)
- `rcs_val.zip` holds the validation JSON/CSV of stages S, C and P only: no buffers, models or test. Send it; it is
  committed under `experiments/results/09_rcs/`.
- `rcs_restore.zip` holds the deployed models A0..A_T, the validation files and a sha256 manifest, with no test.
  Keep it for a new session.

Test is scored once, and only after that commit: set `DECISIONS_COMMIT` in cell 9, then run cells 9-10.

If this session has closed, start a new one and run these cells, never cells 3-7:
1. cell 1;
2. cell 8, with `RESTORE = True` and the restore zip attached;
3. cell 2;
4. cells 9 and 10.

In [ ]:
# 7. Validation outputs: rcs_val.zip (JSON/CSV only) and rcs_restore.zip (deployed models + manifest); no test
assert not (RCS / "test").exists(), "test was already scored in this folder"
run("zip-val", *COMMON, "--bundle", WORK.parent / "rcs_val.zip")
run("bundle", *COMMON, "--bundle", WORK.parent / "rcs_restore.zip")

In [ ]:
# 8. RESTORE (only in a NEW session after the old one closed; skip otherwise). Then run cell 2, then cells 9-10.
RESTORE = False
if RESTORE:
    man = [m for m in sorted(glob.glob("/kaggle/input/**/RESTORE_MANIFEST.json", recursive=True))
           if (Path(m).parent / "rcs").is_dir()] or sorted(glob.glob("/kaggle/input/**/rcs_restore.zip", recursive=True))
    assert man, "attach the dataset made from rcs_restore.zip"
    b = Path(man[0])
    run("restore", *COMMON, "--bundle", b.parent if b.name == "RESTORE_MANIFEST.json" else b)
else:
    print("restore skipped (RESTORE = False)")

In [ ]:
# 9. Test, ONCE. Disabled until DECISIONS_COMMIT is the sha of the commit that holds rcs_val.zip's files.
#    score-test refuses unless every validation file here is byte-identical to that commit's copy under
#    experiments/results/09_rcs/ and every deployed model has the hash recorded in rcs/summary.json.
DECISIONS_COMMIT = ""
if DECISIONS_COMMIT:
    subprocess.run(["git", "-C", CODE_DIR, "fetch", "--depth", "50", "origin", "main"], check=True)
    run("score-test", *COMMON, "--decisions-commit", DECISIONS_COMMIT, "--repo", CODE_DIR)
else:
    print("skipped: commit the validation files first, then set DECISIONS_COMMIT")

In [ ]:
# 10. Test files (after cell 9 ran; original or restore session)
tdir = RCS / "test"
if not tdir.exists():
    print("no test scored yet")
else:
    dest = WORK.parent / "rcs_test.zip"
    pick = sorted(p for p in tdir.rglob("*") if p.is_file())
    with zipfile.ZipFile(dest, "w", zipfile.ZIP_DEFLATED) as zf:
        for p in pick:
            zf.write(p, p.relative_to(WORK))
    print(f"{len(pick)} files -> {dest} ({dest.stat().st_size / 1e6:.2f} MB). Download it from the Output panel.")